In [ ]:

# ============================================================
# 1. LECTURA DE ARCHIVOS ORIGINALES
# ============================================================

import pandas as pd
import os

# Leer archivos originales
df_modis = pd.read_csv(
    "/content/fire_archive_M-C61_845464.csv"
)

df_j2 = pd.read_csv(
    "/content/fire_archive_J2V-C2_845470.csv"
)

df_suomi = pd.read_csv(
    "/content/fire_archive_SV-C2_845465.csv"
)

df_j1 = pd.read_csv(
    "/content/fire_archive_J1V-C2_845466.csv"
)

# Mostrar cantidad de registros
print("ARCHIVOS LEÍDOS CORRECTAMENTE")

print("MODIS:", len(df_modis))
print("NOAA-21:", len(df_j2))
print("SUOMI NPP:", len(df_suomi))
print("NOAA-20:", len(df_j1))

# Comprobar dimensiones
print("\nDIMENSIONES")

print("MODIS:", df_modis.shape)
print("NOAA-21:", df_j2.shape)
print("SUOMI NPP:", df_suomi.shape)
print("NOAA-20:", df_j1.shape)



ARCHIVOS LEÍDOS CORRECTAMENTE
MODIS: 195053
NOAA-21: 81262
SUOMI NPP: 437874
NOAA-20: 235647

DIMENSIONES
MODIS: (195053, 15)
NOAA-21: (81262, 15)
SUOMI NPP: (437874, 15)
NOAA-20: (235647, 15)


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type
0,-37.0977,-71.9882,309.8,1.0,1.0,2000-11-01,1511,Terra,MODIS,48,6.03,297.1,6.9,D,0
1,-37.0993,-71.9766,312.7,1.0,1.0,2000-11-01,1511,Terra,MODIS,60,6.03,298.4,9.1,D,0
2,-22.3197,-68.8811,339.3,1.2,1.1,2000-11-03,1455,Terra,MODIS,82,6.03,314.0,32.0,D,2
3,-23.8286,-70.3111,332.0,1.3,1.1,2000-11-03,1455,Terra,MODIS,64,6.03,313.0,21.2,D,2
4,-22.3131,-68.8858,304.6,1.4,1.2,2000-11-05,347,Terra,MODIS,61,6.03,279.9,15.8,N,2


In [13]:

# ============================================================
# 2. LIMPIEZA MODIS
# ============================================================

modis = df_modis.copy()

# ------------------------------------------------------------
# 2.1. Normalizar nombres de columnas
# ------------------------------------------------------------

modis.columns = (
    modis.columns.str.strip().str.lower()
)

# ------------------------------------------------------------
# 2.2. Seleccionar columnas necesarias
# ------------------------------------------------------------

columnas_requeridas = [
    "latitude",
    "longitude",
    "acq_date",
    "acq_time",
    "satellite",
    "instrument",
    "confidence",
    "frp",
    "type"
]

faltantes = [
    col for col in columnas_requeridas
    if col not in modis.columns
]

if faltantes:
    raise ValueError(
        f"Columnas faltantes en MODIS: {faltantes}"
    )

modis = modis[columnas_requeridas].copy()

# ------------------------------------------------------------
# 2.3. Renombrar FRP
# ------------------------------------------------------------

modis = modis.rename(columns={
    "frp": "frp_mw"
})

# ------------------------------------------------------------
# 2.4. Convertir columnas numéricas
# ------------------------------------------------------------

columnas_numericas = [
    "latitude",
    "longitude",
    "confidence",
    "frp_mw",
    "type"
]

for col in columnas_numericas:
    modis[col] = pd.to_numeric(
        modis[col],
        errors="coerce"
    )

modis["type"] = modis["type"].astype("Int64")

# ------------------------------------------------------------
# 2.5. Homologar confidence MODIS con VIIRS
# ------------------------------------------------------------

def clasificar_confidence(valor):

    if pd.isna(valor):
        return pd.NA

    if 0 <= valor < 30:
        return "low"

    elif 30 <= valor < 80:
        return "nominal"

    elif 80 <= valor <= 100:
        return "high"

    else:
        return pd.NA

modis["confidence"] = (
    modis["confidence"]
    .apply(clasificar_confidence)
    .astype("string")
)

# ------------------------------------------------------------
# 2.6. Limpiar columnas de texto
# ------------------------------------------------------------

columnas_texto = [
    "satellite",
    "instrument"
]

for col in columnas_texto:
    modis[col] = (
        modis[col]
        .astype("string")
        .str.strip()
        .str.upper()
    )

# ------------------------------------------------------------
# 2.7. Limpiar fecha
# ------------------------------------------------------------

modis["acq_date"] = pd.to_datetime(
    modis["acq_date"],
    errors="coerce"
)

# ------------------------------------------------------------
# 2.8. Normalizar y validar hora
# ------------------------------------------------------------

modis["acq_time"] = (
    modis["acq_time"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(4)
)

formato_hora = modis["acq_time"].str.fullmatch(
    r"\d{4}",
    na=False
)

hora_num = pd.to_numeric(
    modis["acq_time"].where(formato_hora),
    errors="coerce"
)

hora_valida = (
    formato_hora
    & (hora_num // 100 < 24)
    & (hora_num % 100 < 60)
)

horas_invalidas = int((~hora_valida).sum())

# ------------------------------------------------------------
# 2.9. Eliminar registros inválidos
# ------------------------------------------------------------

registros_originales = len(modis)

coordenadas_validas = (
    modis["latitude"].between(-90, 90)
    & modis["longitude"].between(-180, 180)
)

fecha_valida = modis["acq_date"].notna()

modis = modis[
    coordenadas_validas
    & fecha_valida
    & hora_valida
].copy()

registros_eliminados_invalidos = (
    registros_originales - len(modis)
)

# ------------------------------------------------------------
# 2.10. Filtrar TYPE = 0
# ------------------------------------------------------------

registros_antes_type = len(modis)

modis = modis[
    modis["type"] == 0
].copy()

registros_eliminados_type = (
    registros_antes_type - len(modis)
)

# ------------------------------------------------------------
# 2.11. Filtrar confidence NOMINAL y HIGH
# ------------------------------------------------------------

registros_antes_confidence = len(modis)

modis = modis[
    modis["confidence"].isin(["nominal", "high"])
].copy()

registros_eliminados_confidence = (
    registros_antes_confidence - len(modis)
)

# ------------------------------------------------------------
# 2.12. Formato final de fecha
# ------------------------------------------------------------

modis["acq_date"] = (
    modis["acq_date"].dt.strftime("%Y-%m-%d")
)

# ------------------------------------------------------------
# 2.13. Eliminar duplicados exactos
# ------------------------------------------------------------

duplicados = int(modis.duplicated().sum())

modis = modis.drop_duplicates().copy()

# ------------------------------------------------------------
# 2.14. Seleccionar columnas finales
# ------------------------------------------------------------

columnas_finales = [
    "latitude",
    "longitude",
    "acq_date",
    "acq_time",
    "satellite",
    "instrument",
    "confidence",
    "frp_mw"
]

modis = modis[columnas_finales].copy()

# ------------------------------------------------------------
# 2.15. Exportar archivo limpio
# ------------------------------------------------------------

os.makedirs("/content/datos_limpios", exist_ok=True)

modis.to_csv(
    "/content/datos_limpios/MODIS_limpio.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 2.16. Informe de limpieza
# ------------------------------------------------------------

print("========== INFORME MODIS ==========")

print("Registros originales:", registros_originales)
print("Eliminados por datos inválidos:", registros_eliminados_invalidos)
print("Eliminados por TYPE != 0:", registros_eliminados_type)
print("Eliminados por confidence:", registros_eliminados_confidence)
print("Horas inválidas originales:", horas_invalidas)
print("Duplicados eliminados:", duplicados)
print("Registros finales:", len(modis))

print("\nDistribución de confidence:")
display(modis["confidence"].value_counts())

print("\nDistribución por satélite:")
display(modis["satellite"].value_counts())

print("\nArchivo generado: MODIS_limpio.csv")

display(modis.head())

print("Dimensiones finales:", modis.shape)
print("Columnas finales:", modis.columns.tolist())


========== INFORME MODIS ==========
Registros originales: 195053
Eliminados por datos inválidos: 0
Eliminados por TYPE != 0: 49706
Eliminados por confidence: 6641
Horas inválidas originales: 0
Duplicados eliminados: 0
Registros finales: 138706

Distribución de confidence:


,count
confidence,
nominal,82944
high,55762



Distribución por satélite:


,count
satellite,
AQUA,97324
TERRA,41382



Archivo generado: MODIS_limpio.csv


,latitude,longitude,acq_date,acq_time,satellite,instrument,confidence,frp_mw
0,-37.0977,-71.9882,2000-11-01,1511,TERRA,MODIS,nominal,6.9
1,-37.0993,-71.9766,2000-11-01,1511,TERRA,MODIS,nominal,9.1
6,-53.0085,-69.2269,2000-11-07,1439,TERRA,MODIS,nominal,5.4
8,-35.8858,-72.6163,2000-11-08,1517,TERRA,MODIS,nominal,7.2
16,-28.2195,-69.6724,2000-11-12,1450,TERRA,MODIS,nominal,15.8


Dimensiones finales: (138706, 8)
Columnas finales: ['latitude', 'longitude', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'frp_mw']


In [14]:

# ============================================================
# 3. LIMPIEZA Y CONSOLIDACIÓN VIIRS
# ============================================================

# ------------------------------------------------------------
# 3.1. Preparar archivos
# ------------------------------------------------------------

archivos_viirs = [
    (df_j2, "fire_archive_J2V-C2_845470.csv"),
    (df_suomi, "fire_archive_SV-C2_845465.csv"),
    (df_j1, "fire_archive_J1V-C2_845466.csv")
]

lista_viirs = []
informe = []

# ------------------------------------------------------------
# 3.2. Procesar cada archivo
# ------------------------------------------------------------

for df_original, nombre_archivo in archivos_viirs:

    print("\nProcesando:", nombre_archivo)

    viirs = df_original.copy()

    registros_originales = len(viirs)

    # --------------------------------------------------------
    # 3.2.1. Normalizar nombres de columnas
    # --------------------------------------------------------

    viirs.columns = (
        viirs.columns.str.strip().str.lower()
    )

    # --------------------------------------------------------
    # 3.2.2. Seleccionar columnas necesarias
    # --------------------------------------------------------

    columnas_requeridas = [
        "latitude",
        "longitude",
        "acq_date",
        "acq_time",
        "satellite",
        "instrument",
        "confidence",
        "frp",
        "type"
    ]

    faltantes = [
        col for col in columnas_requeridas
        if col not in viirs.columns
    ]

    if faltantes:
        raise ValueError(
            f"Columnas faltantes en {nombre_archivo}: {faltantes}"
        )

    viirs = viirs[columnas_requeridas].copy()

    # --------------------------------------------------------
    # 3.2.3. Renombrar FRP
    # --------------------------------------------------------

    viirs = viirs.rename(columns={
        "frp": "frp_mw"
    })

    # --------------------------------------------------------
    # 3.2.4. Convertir columnas numéricas
    # --------------------------------------------------------

    columnas_numericas = [
        "latitude",
        "longitude",
        "frp_mw",
        "type"
    ]

    for col in columnas_numericas:
        viirs[col] = pd.to_numeric(
            viirs[col],
            errors="coerce"
        )

    viirs["type"] = viirs["type"].astype("Int64")

    # --------------------------------------------------------
    # 3.2.5. Limpiar columnas de texto
    # --------------------------------------------------------

    columnas_texto = [
        "satellite",
        "instrument"
    ]

    for col in columnas_texto:
        viirs[col] = (
            viirs[col]
            .astype("string")
            .str.strip()
            .str.upper()
        )

    viirs["confidence"] = (
        viirs["confidence"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    # --------------------------------------------------------
    # 3.2.6. Limpiar fecha
    # --------------------------------------------------------

    viirs["acq_date"] = pd.to_datetime(
        viirs["acq_date"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # 3.2.7. Normalizar y validar hora
    # --------------------------------------------------------

    viirs["acq_time"] = (
        viirs["acq_time"]
        .astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(4)
    )

    formato_hora = viirs["acq_time"].str.fullmatch(
        r"\d{4}",
        na=False
    )

    hora_num = pd.to_numeric(
        viirs["acq_time"].where(formato_hora),
        errors="coerce"
    )

    hora_valida = (
        formato_hora
        & (hora_num // 100 < 24)
        & (hora_num % 100 < 60)
    )

    horas_invalidas = int((~hora_valida).sum())

    # --------------------------------------------------------
    # 3.2.8. Eliminar registros inválidos
    # --------------------------------------------------------

    coordenadas_validas = (
        viirs["latitude"].between(-90, 90)
        & viirs["longitude"].between(-180, 180)
    )

    fecha_valida = viirs["acq_date"].notna()

    viirs = viirs[
        coordenadas_validas
        & fecha_valida
        & hora_valida
    ].copy()

    registros_eliminados_invalidos = (
        registros_originales - len(viirs)
    )

    # --------------------------------------------------------
    # 3.2.9. Filtrar TYPE = 0
    # --------------------------------------------------------

    registros_antes_type = len(viirs)

    viirs = viirs[
        viirs["type"] == 0
    ].copy()

    registros_eliminados_type = (
        registros_antes_type - len(viirs)
    )

    # --------------------------------------------------------
    # 3.2.10. Filtrar confidence NOMINAL y HIGH
    # --------------------------------------------------------

    registros_antes_confidence = len(viirs)

    viirs = viirs[
        viirs["confidence"].isin(["n", "h"])
    ].copy()

    registros_eliminados_confidence = (
        registros_antes_confidence - len(viirs)
    )

    # --------------------------------------------------------
    # 3.2.11. Homologar confidence
    # --------------------------------------------------------

    viirs["confidence"] = viirs["confidence"].map({
        "n": "nominal",
        "h": "high"
    }).astype("string")

    # --------------------------------------------------------
    # 3.2.12. Formato final de fecha
    # --------------------------------------------------------

    viirs["acq_date"] = (
        viirs["acq_date"].dt.strftime("%Y-%m-%d")
    )

    # --------------------------------------------------------
    # 3.2.13. Eliminar duplicados exactos
    # --------------------------------------------------------

    duplicados = int(viirs.duplicated().sum())

    viirs = viirs.drop_duplicates().copy()

    lista_viirs.append(viirs)

    # --------------------------------------------------------
    # 3.2.14. Registrar informe
    # --------------------------------------------------------

    informe.append({
        "archivo": nombre_archivo,
        "registros_originales": registros_originales,
        "eliminados_invalidos": registros_eliminados_invalidos,
        "eliminados_type": registros_eliminados_type,
        "eliminados_confidence": registros_eliminados_confidence,
        "horas_invalidas": horas_invalidas,
        "duplicados_eliminados": duplicados,
        "registros_finales": len(viirs)
    })

# ------------------------------------------------------------
# 3.3. Consolidar los tres archivos
# ------------------------------------------------------------

viirs_limpio = pd.concat(
    lista_viirs,
    ignore_index=True,
    sort=False
)

# ------------------------------------------------------------
# 3.4. Eliminar duplicados del consolidado
# ------------------------------------------------------------

duplicados_consolidados = int(
    viirs_limpio.duplicated().sum()
)

viirs_limpio = viirs_limpio.drop_duplicates().copy()

# ------------------------------------------------------------
# 3.5. Seleccionar columnas finales
# ------------------------------------------------------------

columnas_finales = [
    "latitude",
    "longitude",
    "acq_date",
    "acq_time",
    "satellite",
    "instrument",
    "confidence",
    "frp_mw"
]

viirs_limpio = viirs_limpio[columnas_finales].copy()

# ------------------------------------------------------------
# 3.6. Exportar archivo limpio
# ------------------------------------------------------------

viirs_limpio.to_csv(
    "/content/datos_limpios/VIIRS_limpio.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 3.7. Informe de limpieza
# ------------------------------------------------------------

df_informe = pd.DataFrame(informe)

print("========== INFORME POR ARCHIVO ==========")

display(df_informe)

print("\n========== INFORME GENERAL VIIRS ==========")

print(
    "Registros originales:",
    df_informe["registros_originales"].sum()
)

print(
    "Eliminados por datos inválidos:",
    df_informe["eliminados_invalidos"].sum()
)

print(
    "Eliminados por TYPE != 0:",
    df_informe["eliminados_type"].sum()
)

print(
    "Eliminados por confidence:",
    df_informe["eliminados_confidence"].sum()
)

print(
    "Duplicados eliminados por archivo:",
    df_informe["duplicados_eliminados"].sum()
)

print(
    "Duplicados adicionales del consolidado:",
    duplicados_consolidados
)

print("Registros finales:", len(viirs_limpio))

print("\nDistribución de confidence:")
display(viirs_limpio["confidence"].value_counts())

print("\nDistribución por satélite:")
display(viirs_limpio["satellite"].value_counts())

print("\nArchivo generado: VIIRS_limpio.csv")

display(viirs_limpio.head())

print("Dimensiones finales:", viirs_limpio.shape)
print("Columnas finales:", viirs_limpio.columns.tolist())



Procesando: fire_archive_J2V-C2_845470.csv

Procesando: fire_archive_SV-C2_845465.csv

Procesando: fire_archive_J1V-C2_845466.csv
========== INFORME POR ARCHIVO ==========


,archivo,registros_originales,eliminados_invalidos,eliminados_type,eliminados_confidence,horas_invalidas,duplicados_eliminados,registros_finales
0,fire_archive_J2V-C2_845470.csv,81262,0,12313,9754,0,0,59195
1,fire_archive_SV-C2_845465.csv,437874,0,81221,33130,0,0,323523
2,fire_archive_J1V-C2_845466.csv,235647,0,39578,22641,0,0,173428



========== INFORME GENERAL VIIRS ==========
Registros originales: 754783
Eliminados por datos inválidos: 0
Eliminados por TYPE != 0: 133112
Eliminados por confidence: 65525
Duplicados eliminados por archivo: 0
Duplicados adicionales del consolidado: 0
Registros finales: 556146

Distribución de confidence:


,count
confidence,
nominal,509138
high,47008



Distribución por satélite:


,count
satellite,
SNPP,323523
N20,173428
N21,59195



Archivo generado: VIIRS_limpio.csv


,latitude,longitude,acq_date,acq_time,satellite,instrument,confidence,frp_mw
0,-38.16999,-72.40671,2023-03-01,0448,N21,VIIRS,nominal,0.61
1,-38.17574,-72.40968,2023-03-01,0448,N21,VIIRS,nominal,0.61
2,-38.15250,-73.37381,2023-03-01,0448,N21,VIIRS,nominal,5.07
3,-38.15446,-73.36742,2023-03-01,0448,N21,VIIRS,nominal,5.07
4,-38.39668,-73.38484,2023-03-01,0448,N21,VIIRS,nominal,3.57


Dimensiones finales: (556146, 8)
Columnas finales: ['latitude', 'longitude', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'frp_mw']
